In [ ]:
## imports module
import anndata as ad
import scanpy as sc
import gc, os
import sys
from rich import print
import numpy as np
import psutil
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sea
import harmonypy as hm
import os
import psutil, gc

from rich import print

In [ ]:
## general setting
sc.settings.verbosity = 0
sc.settings.set_figure_params(dpi=80)
pd.set_option('display.max_columns', None)

# check how many memory used
print(psutil.virtual_memory())
# Force garbage collection
gc.collect()  
sc.settings.set_figure_params(dpi=200, facecolor="white")
PATH = "/home/kibr/Work/Vascular_atlas"

## set path and parameters
os.chdir(PATH)
sc.set_figure_params(figsize=(6, 6), frameon=False)

In [ ]:
## --- Load data --- ##
## Setting one: all endothelial cells
adata = sc.read_h5ad(PATH + "/Results/Revision_2/clean_object_revision_03032026.h5ad")#vnoised',color="cell.class_after_decontX", legend_loc="on data")
adata = adata[adata.obs['Cell_class'].isin(['Fibroblast'])].copy()
## Show the data
print(adata)
print(adata.X[:10,:10])

In [ ]:
## save backup
adata.raw = adata.copy()

## filter the features again
sc.pp.filter_genes(adata, min_cells=20) 

print(adata)
print(adata.X[:10,:10])

### Performing another round of integration using Harmony

In [ ]:
## normalization
sc.pp.normalize_total(adata,target_sum=1e6)
sc.pp.log1p(adata)
adata.layers["log1p"] = adata.X.copy()

print(adata.X[:8,:8])

In [ ]:
## to reach the best results, using the below setting
sc.pp.highly_variable_genes(
    adata,
    n_top_genes=2000,
    subset=False,
    layer="log1p",
    flavor="seurat",
    batch_key="individualID",
)

## only using the highly variable genes for integration task
adata = adata[:,adata.var["highly_variable"]].copy()
print(adata)

In [ ]:
### pca and integration
print("Running PCA")
sc.pp.scale(adata, max_value=10)
sc.tl.pca(adata, n_comps=50, svd_solver="arpack")
print(psutil.virtual_memory())

In [ ]:
## Harmony integration 
print(adata)
print("Applying Harmony integration")
sc.external.pp.harmony_integrate(adata, key='individualID', max_iter_harmony=50,basis='X_pca')
rep = "X_pca_harmony"

print(psutil.virtual_memory())

In [ ]:
print("Computing neighbors")
rep = "X_pca_harmony"
sc.pp.neighbors(adata, use_rep=rep,metric="cosine")

print("Computing leiden")
sc.tl.leiden(adata,resolution=1.0,key_added='leiden_harmony')

In [ ]:
# Check integration figures for remaining batch(individual effect)
sc.settings.figdir = f"{PATH}/Results/Revision_2/Figures"
os.makedirs(sc.settings.figdir, exist_ok=True)

sc.tl.umap(
    adata,
    # reuse existing neighbors graph
    random_state=42,
    key_added="umap_harmony"
)

sc.pl.embedding(
    adata,
    basis="umap_harmony", 
    color=["leiden_harmony", "individualID",'brain_region'],
    # color=["leiden_harmony", "individualID",'brain_region','Cell_type'],
    frameon=False,
    ncols=3,
    size=20,
    legend_loc="on data",
    save=f"_Fibroblast_umap_harmony.svg"
)

In [ ]:
### Given some known cell type markers, annotate the clusters
adata = adata.raw.to_adata()
adata.raw = adata.copy()
print(adata.X[:10,:10])

sc.pp.normalize_total(adata,target_sum=1e6)
sc.pp.log1p(adata)
# sc.pp.scale(adata, max_value=10)

markers = ['KCNT2','LAMA1','LAMA2','LAMB1','LAMB2','CEMIP','TRPM3','SLC47A1',
           'RNF220','IGFBP6','CLU','CLDN11','S100B','RBP4',
           'COL1A1','COL1A2','COL3A1','COL5A1','COL1A1','COL1A2','COL3A1','COL5A1',"DCN",'LUM','C3','SCUBE1','MYRIP','SLC4A4',"STXBP6",
           'STAT3','IL4R','NFKB1',"TAGLN","MYH11"]
# markers = {'1.1':"KCNT2",'1.2':"LAMA2",'1.3':'EPS8','2.1':"SLC4A4",'2.2':"SLIT2",'2.3':'SLC26A2','3.1':"KCNIP1",'3.2':"RNF220",'3.3':"LAMC3",'4.1':"CARMN",'4.2':"RGS6",'4.3':'PRKG1','5.1':'TRPM3','5.2':'SLC13A3'}
sc.pl.dotplot(adata, markers, groupby='leiden_harmony',use_raw=False, save=f"_Fibroblast_markers_dotplot.svg")

In [ ]:
## Checking expression of some known markers in the dotplot to annotate the clusters
sc.pl.embedding(
    adata,
    basis="umap_harmony",
    color=['CEMIP','LAMA2','SLC47A1','IGFBP6','CLU','CLDN11','TJP1','TRPM3','RNF220','CDH1','PDGFRA','C3','SCUBE1','MYRIP','SLC4A4',"STXBP6",'STAT3','IL4R','NFKB1'],
    frameon=False,
    use_raw=False,
    ncols=5,
    size=20,
    legend_loc="on data",
    # save=f"_Fibroblast_umap_markers_harmony.svg"
)

In [ ]:
### wilcox test to find the top markers for each cluster
sc.tl.rank_genes_groups(adata, groupby='leiden_harmony', method='wilcoxon', key_added='rank_genes_harmony')
### show the top 10 markers for each cluster
result = adata.uns['rank_genes_harmony']
groups = result['names'].dtype.names
top_markers = pd.DataFrame({group: result['names'][group][:10] for group in groups})
print(top_markers)

In [ ]:
sc.pl.rank_genes_groups_dotplot(
    adata,
    n_genes=5,
    key='rank_genes_harmony',
    groupby='leiden_harmony',
    standard_scale='var',      # 'var' (per gene) or 'obs' (per cell)
    var_group_rotation=45,     # rotate gene labels
    colorbar_title='Mean\nexpression',
    size_title='Fraction\nexpressing',
    figsize=(15, 7),
    # save='_marker_dotplot.pdf'  # optional save
)

In [ ]:
## make the results of the wilcox test into a dataframe including the p value, adjusted p-values and log fold changes
wilcox_results = []
for cluster in groups:
    for gene, pval, pval_adj, logfc in zip(result['names'][cluster], result['pvals'][cluster], result['pvals_adj'][cluster], result['logfoldchanges'][cluster]):
        wilcox_results.append({'cluster': cluster, 'gene': gene, 'pval': pval, 'pval_adj': pval_adj, 'logfc': logfc})
wilcox_df = pd.DataFrame(wilcox_results)
display(wilcox_df.head())

## showing the top 20 markers with smallest adjusted p-values in dataframe for each cluster and the log fold changes should be greater than 0.5
top20_markers_df = wilcox_df[wilcox_df['logfc'] > 0.5].groupby('cluster').apply(lambda x: x.nsmallest(20, 'pval_adj')).reset_index(drop=True)
display(top20_markers_df)

top20_markers_df.to_csv(f"{PATH}/Results/Revision_2/DEG/Fibroblast_top20_markers_harmony.csv", index=False)

In [ ]:
### Try to cluster the groups and run DEG again for checking
cluster2celltype = {
    "0": "Fib_1",
    "1": "Fib_1",
    "2": "Fib_2",
    "3": "Fib_2",
    "4": "Fib_2",
    "5": "Fib_2",
    "6": "Fib_3",
    "7": "Fib_4",
    "8": "Fib_3",
    "9": "Fib_3",
    "10": "Fib_5",
    "11": "Fib_2",
    "12": "Fib_6",
    "13": "Fib_3",
    "14": "Fib_3", 
}
adata.obs['Cell_type'] = adata.obs['leiden_harmony'].map(cluster2celltype)

sc.pl.embedding(
    adata,
    basis='umap_harmony',   # <- THIS picks .obsm['X_' + key]
    # color=['leiden_harmony','Cell_type'],
    color = ['brain_region','region_layer','Cell_type'],
    frameon=False,
    ncols=1,
    size=20,
    # legend_loc="on data",
    use_raw = False,
    # save=f"_Fibroblast_umap_CT_RG.svg"
    )

In [ ]:
### Given some known cell type markers, annotate the clusters
adata = adata.raw.to_adata()
adata.raw = adata.copy()
print(adata.X[:10,:10])

sc.pp.normalize_total(adata,target_sum=1e6)
sc.pp.log1p(adata)

markers = {
    'Perivascular_fibroblast': ['LAMA2','TJP1','PDGFRA'],
    'Meningeal_fibroblast': ['SLC4A4','SLC47A1'],
    'Large_vessel_fibroblast': ['DCN','COL1A2','COL3A1'],
    'ChP_fibroblast': ['MYRIP',"STXBP6",'STAT3','IL4R'],
    'Base_barrier_cell': ['IGFBP6','CLDN11']
    }

sc.pl.dotplot(adata, markers, groupby='Cell_type',use_raw=False,)# save=f"_Fibroblast_markers_dotplot.svg")

In [ ]:
### Given some known cell type markers, annotate the clusters
adata = adata.raw.to_adata()
adata.raw = adata.copy()
print(adata.X[:10,:10])

sc.pp.normalize_total(adata,target_sum=1e6)
sc.pp.log1p(adata)
# sc.pp.scale(adata, max_value=10)

markers = ['KCNT2','LAMA1','LAMA2','LAMB1','LAMB2','CEMIP','TRPM3','SLC47A1',
           'RNF220','IGFBP6','CLU','CLDN11','S100B','RBP4',
           'COL1A1','COL1A2','COL3A1','COL5A1','COL1A1','COL1A2','COL3A1','COL5A1',"DCN",'LUM','C3','SCUBE1','MYRIP','SLC4A4',"STXBP6",
           'STAT3','IL4R','NFKB1']
# markers = {'1.1':"KCNT2",'1.2':"LAMA2",'1.3':'EPS8','2.1':"SLC4A4",'2.2':"SLIT2",'2.3':'SLC26A2','3.1':"KCNIP1",'3.2':"RNF220",'3.3':"LAMC3",'4.1':"CARMN",'4.2':"RGS6",'4.3':'PRKG1','5.1':'TRPM3','5.2':'SLC13A3'}
sc.pl.dotplot(adata, markers, groupby='leiden_harmony',use_raw=False, save=f"_Fibroblast_markers_dotplot.svg")

In [ ]:
### ---------------------------------------------------------------------
### Saving processed data
### ---------------------------------------------------------------------
adata = adata.raw.to_adata()

print(adata)
print(adata.X[:10,:10])
## Saving h5ad
results_file = PATH+"/Results/Revision_2/Fibroblast_temp_processed.h5ad"
adata.write(results_file)
print("Done. Saved:", results_file)

In [ ]:
## check the proportion of clusters across different brain regions
cluster_region_counts = pd.crosstab(adata.obs['Cell_type'], adata.obs['region_layer'])
cluster_region_proportions = cluster_region_counts.div(cluster_region_counts.sum(axis=1), axis=0)
# print(cluster_region_proportions)

## plotting the stacked cluster proportions across brain regions, x asix is the brain regions, y axis is the proportion of cells in each cluster 
## also remove the coordinate line in the frame
ax1 = cluster_region_counts.T.plot(kind='bar', stacked=True, figsize=(8, 2), edgecolor='none')
plt.xlabel('Brain Region')
plt.ylabel('Cell Count')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax1.grid(False)
## similar plot but with each column sum up as 1; put the figure legend outside the plot
cluster_region_proportions_normalized = cluster_region_counts.div(cluster_region_counts.sum(axis=0), axis=1)
ax2 = cluster_region_proportions_normalized.T.plot(kind='bar', stacked=True, figsize=(8, 2), edgecolor='none')
plt.xlabel('Brain Region')
plt.ylabel('Proportion')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax2.grid(False)

In [ ]:
exit()